# 02 — Exploratory Data Analysis

Explores `cohort.csv` and `features_hourly.csv` produced by `01_cohort_extraction.ipynb`.

Sections:
1. Cohort overview & label balance  
2. Patient demographics  
3. ICU stay characteristics  
4. Delirium onset timing  
5. Feature coverage & missingness  
6. Feature value distributions  
7. Temporal density of observations

In [1]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # headless rendering — swap to 'inline' in interactive session
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from pathlib import Path

np.random.seed(42)

OUTPUT_DIR = Path("/users/syang195/1595-final")
FIG_DIR    = OUTPUT_DIR / "results" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

cohort   = pd.read_csv(OUTPUT_DIR / "cohort.csv", parse_dates=["intime", "outtime", "first_delirium_time"])
features = pd.read_csv(OUTPUT_DIR / "features_hourly.csv")

print(f"cohort shape   : {cohort.shape}")
print(f"features shape : {features.shape}")
cohort.head(2)

cohort shape   : (60132, 14)
features shape : (45606339, 4)


,subject_id,hadm_id,stay_id,intime,outtime,los_hours,anchor_age,gender,race,insurance,marital_status,first_careunit,label,first_delirium_time
0,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,93.438056,86,F,WHITE,Medicare,WIDOWED,Medical Intensive Care Unit (MICU),0,NaT
1,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,26.832778,55,F,WHITE,Private,MARRIED,Surgical Intensive Care Unit (SICU),0,NaT


## 1. Cohort Overview & Label Balance

In [ ]:
n_total   = len(cohort)
n_pos     = cohort["label"].sum()
n_neg     = n_total - n_pos
prev      = n_pos / n_total * 100

print(f"Total ICU stays      : {n_total:,}")
print(f"  Delirium (label=1) : {n_pos:,}  ({prev:.1f}%)")
print(f"  Stable   (label=0) : {n_neg:,}  ({100-prev:.1f}%)")
print()
print(f"Unique feature names : {features['feature_name'].nunique()}")
print(f"Feature rows total   : {len(features):,}")

# Label pie chart
fig, ax = plt.subplots(figsize=(4, 4))
ax.pie(
    [n_pos, n_neg],
    labels=[f"Delirium\n{n_pos:,}", f"Stable\n{n_neg:,}"],
    colors=["#DD8452", "#4C72B0"],
    autopct="%1.1f%%",
    startangle=90,
)
ax.set_title("Label Balance")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_label_balance.png", dpi=120)
plt.show()

## 2. Patient Demographics

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Age distribution by label
ax1 = fig.add_subplot(gs[0, :])
for lbl, grp, col in [
    (0, cohort[cohort.label == 0], "#4C72B0"),
    (1, cohort[cohort.label == 1], "#DD8452"),
]:
    ax1.hist(
        grp["anchor_age"].dropna(), bins=30, alpha=0.6,
        color=col, label="Stable" if lbl == 0 else "Delirium", density=True,
    )
ax1.set_xlabel("Age (years)")
ax1.set_ylabel("Density")
ax1.set_title("Age Distribution by Label")
ax1.legend()

# Gender delirium rate
ax2 = fig.add_subplot(gs[1, 0])
gr = (
    cohort.groupby("gender")["label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "delirium_rate", "count": "n"})
)
gr["delirium_rate"].plot(kind="bar", ax=ax2, color=["#4C72B0", "#DD8452"], edgecolor="white", rot=0)
ax2.set_title("Delirium Rate by Gender")
ax2.set_ylabel("Delirium Rate")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for i, (_, row) in enumerate(gr.iterrows()):
    ax2.text(i, row["delirium_rate"] + 0.003, f"n={row['n']:,}", ha="center", fontsize=8)

# Race (top 6)
ax3 = fig.add_subplot(gs[1, 1])
rr = (
    cohort.groupby("race")["label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "delirium_rate", "count": "n"})
    .nlargest(6, "n")
)
rr["delirium_rate"].plot(kind="barh", ax=ax3, color="#55A868", edgecolor="white")
ax3.set_title("Delirium Rate by Race (top 6)")
ax3.set_xlabel("Delirium Rate")
ax3.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax3.tick_params(axis="y", labelsize=7)

# Insurance
ax4 = fig.add_subplot(gs[1, 2])
ir = (
    cohort.groupby("insurance")["label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "delirium_rate", "count": "n"})
)
ir["delirium_rate"].sort_values().plot(kind="barh", ax=ax4, color="#C44E52", edgecolor="white")
ax4.set_title("Delirium Rate by Insurance")
ax4.set_xlabel("Delirium Rate")
ax4.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.suptitle("Patient Demographics", fontsize=14, fontweight="bold", y=1.01)
plt.savefig(FIG_DIR / "02_demographics.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nAge summary by label:")
print(cohort.groupby("label")["anchor_age"].describe().round(1).to_string())

## 3. ICU Stay Characteristics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# LOS distribution (capped at 30 days for readability)
ax = axes[0]
los_cap = 30 * 24  # hours
for lbl, grp, col in [
    (0, cohort[cohort.label == 0], "#4C72B0"),
    (1, cohort[cohort.label == 1], "#DD8452"),
]:
    los = grp["los_hours"].clip(upper=los_cap)
    ax.hist(
        los / 24, bins=40, alpha=0.6, color=col,
        label="Stable" if lbl == 0 else "Delirium", density=True,
    )
ax.set_xlabel("ICU LOS (days, capped at 30)")
ax.set_ylabel("Density")
ax.set_title("ICU Length of Stay by Label")
ax.legend()

# Care unit mix
ax2 = axes[1]
if "first_careunit" in cohort.columns:
    unit_counts = cohort["first_careunit"].value_counts()
    unit_counts.plot(kind="barh", ax=ax2, color="#8172B3", edgecolor="white")
    ax2.set_title("Stays by First Care Unit")
    ax2.set_xlabel("Number of stays")
    ax2.tick_params(axis="y", labelsize=8)
else:
    ax2.text(0.5, 0.5, "first_careunit not available", ha="center", va="center")
    ax2.set_title("Care Unit")

plt.tight_layout()
plt.savefig(FIG_DIR / "03_icu_stay.png", dpi=120)
plt.show()

print("\nICU LOS summary (hours) by label:")
print(cohort.groupby("label")["los_hours"].describe().round(1).to_string())

## 4. Delirium Onset Timing

In [ ]:
del_cohort = cohort[cohort["label"] == 1].copy()
del_cohort["hours_to_delirium"] = (
    (del_cohort["first_delirium_time"] - del_cohort["intime"]).dt.total_seconds() / 3600
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of onset time
ax = axes[0]
del_cohort["hours_to_delirium"].clip(upper=14 * 24).hist(
    bins=48, ax=ax, color="#DD8452", edgecolor="white"
)
ax.set_xlabel("Hours from ICU admission to first positive CAM-ICU")
ax.set_ylabel("Count")
ax.set_title("Delirium Onset Timing (capped at 14 days)")

# Cumulative onset curve
ax2 = axes[1]
onset_sorted = del_cohort["hours_to_delirium"].dropna().sort_values()
cum_frac     = np.arange(1, len(onset_sorted) + 1) / len(onset_sorted)
ax2.plot(onset_sorted / 24, cum_frac, color="#DD8452", linewidth=2)
ax2.set_xlabel("Days from ICU admission")
ax2.set_ylabel("Cumulative fraction of delirium cases")
ax2.set_title("Cumulative Delirium Onset Curve")
ax2.axvline(x=1, color="grey", linestyle="--", linewidth=0.8, label="Day 1")
ax2.axvline(x=3, color="grey", linestyle=":",  linewidth=0.8, label="Day 3")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "04_delirium_timing.png", dpi=120)
plt.show()

print(f"Median hours to delirium : {del_cohort['hours_to_delirium'].median():.1f}")
print(f"% onset within 24 h      : {(del_cohort['hours_to_delirium'] < 24).mean()*100:.1f}%")
print(f"% onset within 72 h      : {(del_cohort['hours_to_delirium'] < 72).mean()*100:.1f}%")

## 5. Feature Coverage & Missingness

Coverage = fraction of ICU stays that have **at least one** observation for a given feature.

In [ ]:
n_stays = cohort["stay_id"].nunique()

coverage = (
    features.groupby("feature_name")["stay_id"]
    .nunique()
    .sort_values(ascending=True)
    / n_stays
)

fig, ax = plt.subplots(figsize=(9, max(6, len(coverage) * 0.35)))
bars = ax.barh(
    coverage.index, coverage.values * 100,
    color=["#4C72B0" if v >= 0.5 else "#DD8452" for v in coverage.values],
    edgecolor="white",
)
ax.axvline(50, color="black", linestyle="--", linewidth=0.8, label="50% threshold")
ax.set_xlabel("Coverage (% of stays)")
ax.set_title("Feature Coverage Across ICU Stays")
ax.legend()
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height() / 2, f"{w:.0f}%", va="center", fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_feature_coverage.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nFeatures with ≥ 50% coverage : {(coverage >= 0.5).sum()}/{len(coverage)}")
print(f"Features with ≥ 90% coverage : {(coverage >= 0.9).sum()}/{len(coverage)}")

## 6. Feature Value Distributions (Stable vs Delirium)

Median value per stay, split by label — shown for the 14 chart features.

In [ ]:
# Median per stay for each feature
feat_median = (
    features
    .groupby(["stay_id", "feature_name"])["value"]
    .median()
    .reset_index()
)
feat_median = feat_median.merge(cohort[["stay_id", "label"]], on="stay_id", how="left")

# Chart features subset (non-drug)
chart_feat_names = [
    "cam_icu", "rass", "gcs_eye", "gcs_verbal", "gcs_motor",
    "heart_rate", "sbp", "dbp", "map", "spo2",
    "resp_rate", "temperature", "fio2", "peep",
]
chart_data = feat_median[feat_median["feature_name"].isin(chart_feat_names)]

n_feats = len(chart_feat_names)
ncols   = 4
nrows   = (n_feats + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
axes_flat = axes.flat

for fname, ax in zip(chart_feat_names, axes_flat):
    sub = chart_data[chart_data["feature_name"] == fname]
    groups = [sub[sub["label"] == 0]["value"], sub[sub["label"] == 1]["value"]]
    ax.boxplot(groups, labels=["Stable", "Delirium"], patch_artist=True,
               boxprops=dict(facecolor="#D0D8EA"),
               medianprops=dict(color="black", linewidth=1.5))
    ax.set_title(fname, fontsize=9)
    ax.tick_params(axis="both", labelsize=8)

# Hide unused axes
for ax in list(axes_flat)[n_feats:]:
    ax.set_visible(False)

plt.suptitle("Median Feature Values: Stable vs Delirium", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_chart_feature_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Key lab features
lab_feat_names = [
    "lactate", "creatinine", "bun", "glucose",
    "sodium", "potassium", "ph", "bicarbonate",
    "wbc", "hemoglobin", "platelets", "inr",
]
lab_data = feat_median[feat_median["feature_name"].isin(lab_feat_names)]

ncols = 4
nrows = (len(lab_feat_names) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
axes_flat = axes.flat

for fname, ax in zip(lab_feat_names, axes_flat):
    sub = lab_data[lab_data["feature_name"] == fname]
    groups = [sub[sub["label"] == 0]["value"], sub[sub["label"] == 1]["value"]]
    ax.boxplot(groups, labels=["Stable", "Delirium"], patch_artist=True,
               boxprops=dict(facecolor="#D0EAD8"),
               medianprops=dict(color="black", linewidth=1.5))
    ax.set_title(fname, fontsize=9)
    ax.tick_params(axis="both", labelsize=8)

for ax in list(axes_flat)[len(lab_feat_names):]:
    ax.set_visible(False)

plt.suptitle("Median Lab Values: Stable vs Delirium", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "07_lab_feature_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Temporal Density of Observations

In [ ]:
# Number of unique features observed per hour_offset, averaged across stays
feat_group = ["chart", "lab", "drug"]

chart_names = {
    "cam_icu", "rass", "gcs_eye", "gcs_verbal", "gcs_motor",
    "heart_rate", "sbp", "dbp", "map", "spo2",
    "resp_rate", "temperature", "fio2", "peep",
}

def _group(name):
    if name in chart_names:
        return "chart"
    if name.startswith("drug_"):
        return "drug"
    return "lab"

features["group"] = features["feature_name"].apply(_group)

# Cap hour_offset at 7 days for readability
feat_cap = features[features["hour_offset"] <= 7 * 24].copy()

density = (
    feat_cap
    .groupby(["hour_offset", "group"])["stay_id"]
    .count()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 4))
palette = {"chart": "#4C72B0", "lab": "#55A868", "drug": "#DD8452"}
for grp in ["chart", "lab", "drug"]:
    if grp in density.columns:
        ax.plot(
            density.index / 24, density[grp],
            label=grp.capitalize(), color=palette[grp], linewidth=1.2, alpha=0.9,
        )
ax.set_xlabel("Days from ICU admission")
ax.set_ylabel("Observation count (all stays)")
ax.set_title("Temporal Density of Feature Observations (first 7 days)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "08_temporal_density.png", dpi=120)
plt.show()

In [ ]:
# Observation count per feature — bar chart
feat_counts = features["feature_name"].value_counts().sort_values()

colors = [
    "#4C72B0" if f in chart_names
    else ("#DD8452" if f.startswith("drug_") else "#55A868")
    for f in feat_counts.index
]

fig, ax = plt.subplots(figsize=(9, max(6, len(feat_counts) * 0.35)))
ax.barh(feat_counts.index, feat_counts.values, color=colors, edgecolor="white")
ax.set_xlabel("Total observations")
ax.set_title("Observation Count per Feature (blue=chart, green=lab, orange=drug)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{int(x):,}"))
plt.tight_layout()
plt.savefig(FIG_DIR / "09_obs_per_feature.png", dpi=120, bbox_inches="tight")
plt.show()